# Preprocessing Layer: EV Charging Station Recommender
This notebook handles initial data cleaning, EDA, arithmetic feature engineering, and data splitting. The results are saved as `train.csv`, `val.csv`, and `test.csv` for downstream model training.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

# Step 1 — Load & Drop Columns
def load_and_clean_data(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Dataset not found at {filepath}")
    
    df = pd.read_csv(filepath)
    df_working = df.copy()
    
    # Columns to drop immediately as per requirements
    cols_to_drop = [
        'timestamp', 'latitude', 'longitude', 'location_type', 'amenities_nearby',
        'ports_available', 'precipitation_mm', 'temperature_f', 'weather_condition',
        'gas_price_per_gallon', 'local_event', 'is_weekend', 'month'
    ]
    
    return df_working.drop(columns=cols_to_drop)

df = load_and_clean_data('../data/ev_stations.csv')
print(f"Raw dataset loaded and columns dropped. Shape: {df.shape}")

In [ ]:
# Step 2 — Exploratory Data Analysis (EDA)
def run_eda(df):
    print("\n--- EDA REPORT ---")
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    
    print("\nData Types:")
    print(df.dtypes)
    
    print("\nMissing Values:")
    missing = df.isnull().sum()
    print(pd.DataFrame({'Count': missing, 'Percentage': (missing / len(df)) * 100}))
    
    print("\nDescriptive Statistics (Numeric):")
    print(df.describe())
    
    print("\nCategorical Value Counts:")
    for col in df.select_dtypes(include=['object']).columns:
        print(f"\n{col}:")
        print(df[col].value_counts().head(10)) # Top 10 for readability
    
    print("\nCorrelation Matrix (Numeric Features):")
    numeric_df = df.select_dtypes(include=[np.number])
    print(numeric_df.corr())
    
    print("\nSkewness and Kurtosis Summary:")
    stats = pd.DataFrame({
        'Skewness': numeric_df.skew(),
        'Kurtosis': numeric_df.kurtosis()
    })
    print(stats)
    
    print("\nClass Balance (Target: is_peak_hour):")
    print(df['is_peak_hour'].value_counts(normalize=True))
    
    print("\nOutlier Detection (IQR Method):")
    for col in numeric_df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
        if outliers > 0:
            print(f"- {col}: {outliers} outliers identified")

run_eda(df)

### Step 3 — Feature Engineering ONLY (Arithmetic Ratios)

In [ ]:
def compute_arithmetic_ratios(df):
    df_copy = df.copy()
    df_copy['ports_available_ratio'] = (df_copy['ports_total'] - df_copy['ports_occupied']) / df_copy['ports_total']
    df_copy['load_factor'] = df_copy['ports_occupied'] / df_copy['ports_total']
    df_copy['service_ratio'] = df_copy['ports_out_of_service'] / df_copy['ports_total']
    return df_copy

df = compute_arithmetic_ratios(df)
print("Arithmetic ratios computed successfully (ports_available_ratio, load_factor, service_ratio).")

In [ ]:
# Step 4 — Train / Validation / Test Split (70/15/15)
# Using random_state=42 and stratified split on is_peak_hour
train_df, temp_df = train_test_split(
    df, 
    train_size=0.70, 
    random_state=42, 
    stratify=df['is_peak_hour']
)

val_df, test_df = train_test_split(
    temp_df, 
    train_size=0.50, # 15/30 = 0.5
    random_state=42, 
    stratify=temp_df['is_peak_hour']
)

print("Data Split Summary:")
print(f"- Training Set:   {len(train_df)} rows")
print(f"- Validation Set: {len(val_df)} rows")
print(f"- Test Set:       {len(test_df)} rows")

In [ ]:
# Step 5 — Save Splits to Disk
output_dir = '../data/'

train_df.to_csv(os.path.join(output_dir, 'train.csv'), index=False)
val_df.to_csv(os.path.join(output_dir, 'val.csv'), index=False)
test_df.to_csv(os.path.join(output_dir, 'test.csv'), index=False)

print(f"Splits saved successfully to {output_dir}")
print(f"1. {os.path.join(output_dir, 'train.csv')} ({len(train_df)} rows)")
print(f"2. {os.path.join(output_dir, 'val.csv')} ({len(val_df)} rows)")
print(f"3. {os.path.join(output_dir, 'test.csv')} ({len(test_df)} rows)")